# TON external validation: independent metric check
Run from this repository. Reads local ignored experiment output; does not execute retrieval, rewrite queries or alter product state. Gold and manual adjudication were frozen separately. No raw document text is embedded in this notebook.

In [ ]:
from pathlib import Path
import json, hashlib, math, collections
root = Path.cwd()
while not (root / 'docs/product/COMMUNITY_INTELLIGENCE_AI_PRD_V2_1.md').exists():
    assert root != root.parent
    root = root.parent
local = root / 'data/generated/validation/ton-docs-v1'
review = root / 'docs/validation/ton-docs-v1'
rows = json.loads((local / 'baseline-run-1/results.json').read_text())
published = json.loads((review / 'results.json').read_text())
receipt = published['receipt']
assert hashlib.sha256((review / 'queries.json').read_bytes()).hexdigest() == receipt['query_sha256']
for path, digest in receipt['baseline']['product_hashes'].items():
    assert hashlib.sha256((root / path).read_bytes()).hexdigest() == digest
lock = json.loads((local / 'corpus.lock.json').read_text())
for f in lock['files']:
    assert hashlib.sha256((local / 'raw' / f['path']).read_bytes()).hexdigest() == f['sha256']
positive = [r for r in rows if r['question']['gold']]
recalls, rr, ndcg = [], [], []
for r in positive:
    gold = set(r['question']['gold'])
    actual = [c['ref'] for c in r['actual_top_k']][:5]
    gains = [int(c in gold) for c in actual]
    recalls.append(len(gold.intersection(actual)) / len(gold))
    rr.append(next((1/(i+1) for i,g in enumerate(gains) if g), 0))
    dcg = sum(g / math.log2(i+2) for i,g in enumerate(gains))
    ideal = sum(1 / math.log2(i+2) for i in range(min(5,len(gold))))
    ndcg.append(dcg/ideal)
computed = dict(zip(['recall_at_5','mrr_at_5','ndcg_at_5'], [sum(v)/len(v) for v in [recalls,rr,ndcg]]))
for key,value in computed.items():
    assert abs(value-published['metrics']['retrieval'][key]) < 1e-12
assert len(rows)==34 and len(positive)==28
assert sum(len(r['citation_valid']) for r in rows)==98
assert all(all(r['citation_valid']) for r in rows)
print('Verified original query/corpus/product hashes, N=34, N_answerable=28, 98 citations, Recall/MRR/nDCG:', computed)


In [ ]:
chunks = json.loads((local / 'prepared/chunks.json').read_text())
flat = [c for group in chunks.values() for c in group]
texts = [c['text'] for c in flat]
profile = {'sources':len(chunks), 'chunks':len(flat), 'raw_bytes':sum(f['bytes'] for f in lock['files'] if f['path'].startswith('content/')), 'whitespace_tokens_including_overlap':sum(c['token_count'] for c in flat), 'chunks_under_10_tokens':sum(c['token_count']<10 for c in flat), 'chunks_without_section':sum(c['section'] is None for c in flat), 'unique_chunk_texts':len(set(texts)), 'max_chunk_tokens':max(c['token_count'] for c in flat)}
print(profile)
assert profile['max_chunk_tokens'] <= 180
assert lock['spec']['selection_frozen_at'] < lock['retrieved_at'] < receipt['started_at']
assert published['metrics']['grounded_answer_accuracy'] == {'correct':6,'N':28}
print('Read-only validation complete; historical truth remains unavailable.')
